# ISOM 839 · Session 11 — Deep Learning Meets Optimization: Predict, Then Decide

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Two demand models. Model A has the better RMSE. Model B makes the company more money. Tonight lives inside that question. You will build the full **predict-then-optimize** pipeline — a small neural network forecasts demand, a newsvendor turns each forecast into an order, a simulator scores the money — and then watch the leaderboard flip when you change one line: the loss function.

In [ ]:
%pip install -q gurobipy
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
torch.manual_seed(839); np.random.seed(839)
print('torch', torch.__version__)

---
## Part 1 — Synthetic retail demand with three features

A store's daily demand depends on the **day of week**, the **price**, and whether there is a **promotion** — plus noise that is wider on promo days (real promotions are unpredictable). We generate 2,000 days and keep 500 for testing. In the lab you never see the formula; in real life you never see it either.

In [ ]:
def make_data(n, seed):
    rng = np.random.default_rng(seed)
    dow   = rng.integers(0, 7, n)                         # 0 = Monday
    price = rng.uniform(8, 14, n)                         # $
    promo = rng.binomial(1, 0.25, n)
    base  = 60 + 12 * np.isin(dow, [4, 5]) - 4.0 * (price - 11) + 25 * promo
    noise = rng.normal(0, 10 + 18 * promo)                # promo days are much noisier
    demand = np.clip(base + noise, 0, None).round()
    X = np.column_stack([dow / 6, (price - 11) / 3, promo]).astype(np.float32)
    return torch.tensor(X), torch.tensor(demand, dtype=torch.float32).unsqueeze(1), demand

X_train, y_train, d_train = make_data(2000, 1)
X_test,  y_test,  d_test  = make_data(500, 2)
print('train', X_train.shape, '· test', X_test.shape, '· mean demand', d_train.mean().round(1))

---
## Part 2 — Stage 1: a tiny neural network (trained on MSE)

A neural network is a flexible function trained by gradient descent on a loss you choose. Two layers, 16 hidden units, 300 epochs — the whole thing fits on a napkin. Tonight's loss is the one every tutorial uses: **mean squared error**.

In [ ]:
def train(loss_fn, epochs=300, lr=0.02, seed=839):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); loss = loss_fn(net(X_train), y_train); loss.backward(); opt.step()
    return net

def rmse(net):
    with torch.no_grad(): return float(torch.sqrt(((net(X_test) - y_test) ** 2).mean()))

net_mse = train(nn.functional.mse_loss)
print(f'MSE-trained model · test RMSE {rmse(net_mse):.2f} units')

---
## Part 3 — Stage 2: the newsvendor turns a forecast into an order

Unit cost **$4**, selling price **$10**, salvage **$1** for anything unsold. Order too little and you lose $6 of margin per unit; order too much and you lose $3 per unit. That asymmetry is the whole story tonight. For a *point* forecast, the newsvendor LP simply orders the forecast — so the decision inherits every bias of the prediction.

In [ ]:
COST, PRICE, SALVAGE = 4, 10, 1

def order_quantity(d_hat):
    m = gp.Model(); m.Params.OutputFlag = 0
    q, sold, left = m.addVar(name='q'), m.addVar(name='sold'), m.addVar(name='left')
    m.addConstr(sold <= d_hat); m.addConstr(sold <= q); m.addConstr(left == q - sold)
    m.setObjective(PRICE * sold + SALVAGE * left - COST * q, GRB.MAXIMIZE)
    m.optimize()
    return q.X

def profit(q, d):
    sold = min(q, d); return PRICE * sold + SALVAGE * (q - sold) - COST * q

print('forecast 70 ->', order_quantity(70), 'units ordered')

---
## Part 4 — The scoreboard: regret, not RMSE

For every test day: order from the model's forecast, let the real demand arrive, compute profit, and compare with the profit you would have made with **perfect foresight**. The average gap is **decision regret** — money left on the table because the forecast was wrong *in the direction that hurts.*

In [ ]:
def avg_regret(net):
    with torch.no_grad(): d_hat = net(X_test).squeeze(1).clamp(min=0).numpy()
    q = np.array([order_quantity(float(x)) for x in d_hat])
    realized = np.array([profit(qi, di) for qi, di in zip(q, d_test)])
    perfect  = np.array([profit(di, di) for di in d_test])
    return float((perfect - realized).mean()), float(realized.mean())

reg_mse, prof_mse = avg_regret(net_mse)
print(f'MSE model · RMSE {rmse(net_mse):.2f} · avg profit ${prof_mse:.2f}/day · avg regret ${reg_mse:.2f}/day')

---
## Part 5 — Flip the leaderboard: train for the decision

The newsvendor's optimal order is not the *mean* of demand but the **critical-fractile quantile**: (price − cost) / (price − salvage) = 6/9 = **0.667**. So train the network to predict *that quantile* instead of the mean — the **pinball (quantile) loss**, three lines long. Its RMSE will be worse. Watch what happens to the money.

In [ ]:
TAU = (PRICE - COST) / (PRICE - SALVAGE)
def pinball(pred, y, tau=TAU):
    e = y - pred
    return torch.mean(torch.maximum(tau * e, (tau - 1) * e))

net_q = train(pinball)
reg_q, prof_q = avg_regret(net_q)
print(f'{"model":16s} {"RMSE":>7s} {"profit/day":>11s} {"regret/day":>11s}')
print(f'{"MSE loss":16s} {rmse(net_mse):7.2f} {prof_mse:11.2f} {reg_mse:11.2f}')
print(f'{"quantile loss":16s} {rmse(net_q):7.2f} {prof_q:11.2f} {reg_q:11.2f}')
print(f'\nThe quantile model is WORSE on RMSE and BETTER on money by ${reg_mse - reg_q:.2f} per day ≈ ${(reg_mse - reg_q) * 365:,.0f} per year, per store.')

In [ ]:
with torch.no_grad():
    p_mse = net_mse(X_test).squeeze(1).numpy(); p_q = net_q(X_test).squeeze(1).numpy()
plt.figure(figsize=(7, 3.6))
plt.scatter(d_test, p_mse, s=8, alpha=.5, label='MSE model'); plt.scatter(d_test, p_q, s=8, alpha=.5, label='quantile model')
lim = [d_test.min(), d_test.max()]; plt.plot(lim, lim, 'k--', lw=1)
plt.xlabel('realized demand'); plt.ylabel('forecast'); plt.legend(); plt.title('The quantile model deliberately over-forecasts — because under-ordering costs twice as much'); plt.show()

**Read the picture.** The quantile model sits above the 45° line on purpose: a forecast that is "wrong" by over-predicting loses $3 a unit; one that under-predicts loses $6. The MSE model is honest about the mean and expensive about the decision.

### 💬 Discussion
What would you tell a VP who just spent a year and $2M raising forecast accuracy by 5%? Under what cost structures does better RMSE translate into better decisions — and when is it a mirage? *(Hint: set salvage to $7 and re-run. When the two losses are nearly symmetric, the leaderboard stops flipping.)*

### Decision-focused learning, the frontier
We changed the loss to a quantile because the newsvendor's optimal decision has a closed form. For general optimization models it does not — and **decision-focused learning** (Mandi et al., JAIR 2024) trains the predictor by differentiating *through* the optimization problem itself. Same idea, no closed form, much harder math; the business lesson is identical.

---
## Lab extension (optional, no submission)

The Session 11 problem set became the Dec 9 final-project milestone. If you want the practice anyway: train both models at **five** salvage values from $1 to $9, tabulate RMSE and regret for each, and write a paragraph on how your organization should evaluate its forecasting team. Bring it to office hours and we will talk through it.

**Next week — Session 12:** the model stops predicting and starts acting. Q-learning, deployed RL, and the LLM copilot that drafts your model — and why you must verify every line it writes. 🤖